In [38]:
import requests, sys
sys.path.append('backend/app/')
import pandas as pd, geopandas as gpd
from backend.app.services import functions

start = "2026-09-01"
end = "2026-10-01"
time_zone = "Europe/Oslo"

In [2]:
CLIENT_ID = "f6d537e9-cf9d-4a87-a838-02d607fe68bd"
url = "https://frost.met.no/observations/v0.jsonld"
ids = ['SN99950', 'SN423100']
elements = ['air_temperature', 'sum(precipitation_amount PT1H)', 'mean(wind_from_direction PT1H)', 'mean(wind_speed PT1H)']

In [ ]:
all_observations = []
for station_id in ids:
    for element in elements:
        params = {
            "sources": station_id, "elements": element,
            "referencetime": "2026-09-01/2026-10-01",
        }
        try:
            response = requests.get(url, params=params, auth=(CLIENT_ID, ""), timeout=60)
            if response.status_code != 200:
                print(f"Request failed for {station_id} | {element}: {response.status_code}")
                continue
            response.raise_for_status()
            data = response.json()
            records = data.get('data', [])
            if not records: continue
            stations = pd.DataFrame(records)
            stations = stations.explode('observations', ignore_index=True)
            obs = pd.json_normalize(stations['observations'])
            if obs.empty: continue
            result = pd.concat([
                stations[['sourceId', 'referenceTime']].reset_index(drop=True),
                obs.reset_index(drop=True)
            ], axis=1)
            result['Time'] = pd.to_datetime(stations['referenceTime'], utc=True)
            result['sourceId'] = station_id
            result = result[result["timeResolution"] == "PT1H"].copy()
            all_observations.append(result)
            print(f"{station_id} | {element}: {len(result)} records")
        except requests.exceptions.RequestException as e:
            print(f"Request failed for {station_id} | {element}: {e}")
df = (
    pd.concat(all_observations, ignore_index=True) if all_observations else pd.DataFrame()
)
df = df[['Time', 'sourceId', 'elementId', 'value']]

SN99950 | air_temperature: 720 records
SN99950 | sum(precipitation_amount PT1H): 720 records
Request failed for SN99950 | mean(wind_from_direction PT1H): 412
Request failed for SN99950 | mean(wind_speed PT1H): 412
Request failed for SN423100 | air_temperature: 412
Request failed for SN423100 | sum(precipitation_amount PT1H): 412
Request failed for SN423100 | mean(wind_from_direction PT1H): 412
Request failed for SN423100 | mean(wind_speed PT1H): 412


In [34]:
df

,sourceId,referenceTime,elementId,value,timeResolution
0,SN99950:0,2026-09-01 00:00:00+00:00,air_temperature,3.7,PT1H
1,SN99950:0,2026-09-01 01:00:00+00:00,air_temperature,3.5,PT1H
2,SN99950:0,2026-09-01 02:00:00+00:00,air_temperature,3.6,PT1H
3,SN99950:0,2026-09-01 03:00:00+00:00,air_temperature,3.6,PT1H
4,SN99950:0,2026-09-01 04:00:00+00:00,air_temperature,3.4,PT1H
...,...,...,...,...,...
1435,SN99950:0,2026-09-30 19:00:00+00:00,sum(precipitation_amount PT1H),2.3,PT1H
1436,SN99950:0,2026-09-30 20:00:00+00:00,sum(precipitation_amount PT1H),2.6,PT1H
1437,SN99950:0,2026-09-30 21:00:00+00:00,sum(precipitation_amount PT1H),0.1,PT1H
1438,SN99950:0,2026-09-30 22:00:00+00:00,sum(precipitation_amount PT1H),0.0,PT1H


In [35]:
df_wide = df.pivot_table(
    index=["referenceTime", "sourceId"],
    columns="elementId",
    values="value",
    aggfunc="first"
).reset_index()
df_wide

elementId,referenceTime,sourceId,air_temperature,sum(precipitation_amount PT1H)
0,2026-09-01 00:00:00+00:00,SN99950:0,3.7,0.0
1,2026-09-01 01:00:00+00:00,SN99950:0,3.5,0.0
2,2026-09-01 02:00:00+00:00,SN99950:0,3.6,0.0
3,2026-09-01 03:00:00+00:00,SN99950:0,3.6,0.0
4,2026-09-01 04:00:00+00:00,SN99950:0,3.4,0.0
...,...,...,...,...
715,2026-09-30 19:00:00+00:00,SN99950:0,8.3,2.3
716,2026-09-30 20:00:00+00:00,SN99950:0,8.3,2.6
717,2026-09-30 21:00:00+00:00,SN99950:0,8.5,0.1
718,2026-09-30 22:00:00+00:00,SN99950:0,8.7,0.0


In [20]:
params = {
    "sources": "SN99950", "elements": 'air_temperature',
    "referencetime": "2026-09-01/2026-10-01",
}
response = requests.get(url, params=params, auth=(CLIENT_ID, ""))
# response.raise_for_status()
data = response.json()
stations = pd.DataFrame(data['data'])
stations

,sourceId,referenceTime,observations
0,SN99950:0,2026-09-01T00:00:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.7..."
1,SN99950:0,2026-09-01T00:10:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.5..."
2,SN99950:0,2026-09-01T00:20:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.3..."
3,SN99950:0,2026-09-01T00:30:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.5..."
4,SN99950:0,2026-09-01T00:40:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.5..."
...,...,...,...
4315,SN99950:0,2026-09-30T23:10:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.9..."
4316,SN99950:0,2026-09-30T23:20:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.8..."
4317,SN99950:0,2026-09-30T23:30:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.9..."
4318,SN99950:0,2026-09-30T23:40:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.9..."


In [25]:
stations['observations'][0]

[{'elementId': 'air_temperature',
  'value': 3.7,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT1H',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0},
 {'elementId': 'air_temperature',
  'value': 3.7,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT10M',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0}]

In [26]:
stations['observations'][1]

[{'elementId': 'air_temperature',
  'value': 3.5,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT10M',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0}]

In [27]:
stations['observations'][2]

[{'elementId': 'air_temperature',
  'value': 3.3,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT10M',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0}]

In [ ]:
pd.DataFrame(stations)['observations'].iloc[0]

In [ ]:
import requests

url = "https://frost.met.no/observations/availableTimeSeries/v0.jsonld"

params = {
    "sources": "SN47230"
}

response = requests.get(
    url,
    params=params,
    auth=(CLIENT_ID, "")
)

response.raise_for_status()

data = response.json()

print(data.keys())

In [ ]:
for item in data["data"]:
    print(
        "Source:", item.get("sourceId"),
        "| Element:", item.get("elementId"),
        "| Unit:", item.get("unit"),
        "| Resolution:", item.get("timeResolution"),
        "| From:", item.get("validFrom"),
        "| To:", item.get("validTo")
    )

In [ ]:
df.columns

In [ ]:
df['geometry'] = df['geometry'].apply(lambda x: x['coordinates'] if isinstance(x, dict) and 'coordinates' in x else None)

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['geometry'].apply(lambda x: x[0] if x else None), df['geometry'].apply(lambda x: x[1] if x else None)), crs='EPSG:4326')

In [ ]:
gdf

In [ ]:
gdf.to_file("met_stations.geojson", driver='GeoJSON')